# Maximum likelihood: a 1D Gaussian

Can you fit a bell curve to a set of observations by eye—and then improve the fit using likelihood?

**Run all cells**, then adjust the **mean** $\mu$ and **variance** $\sigma^2$. The plot and likelihood update when you release a slider. The data stay fixed while you explore. 

Our model is $x_i\sim\mathcal{N}(\mu,\sigma^2)$, independently. The mean controls the center; the standard deviation $\sigma=\sqrt{\sigma^2}$ controls the width.

The histogram shows observed density, and the short marks along the horizontal axis show the individual observations. The orange curve is the chosen Gaussian density. Likelihood uses the individual observations, so it does not depend on the histogram bins.

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

# Change these settings and run all cells to explore another sample.
seed = 42
n = 60
true_mean = 1.5
true_variance = 1.2
rng = np.random.default_rng(seed)
points = rng.normal(loc=true_mean, scale=np.sqrt(true_variance), size=n)


## Scoring a fit

For independent observations, the likelihood is the product of the densities at the observed values:

$$L(\mu,\sigma^2)=\prod_{i=1}^n\frac{1}{\sqrt{2\pi\sigma^2}}
\exp\!\left(-\frac{(x_i-\mu)^2}{2\sigma^2}\right).$$

We compute the log-likelihood to avoid multiplying many small numbers:

$$\ell(\mu,\sigma^2)=-\frac{n}{2}\log(2\pi\sigma^2)
-\frac{1}{2\sigma^2}\sum_{i=1}^n(x_i-\mu)^2.$$

Maximizing $\ell$ gives the same fit as maximizing $L$. **Higher is better**, including when the values are negative. Likelihood is a product of *densities*, not the probability of observing these exact continuous-valued points, and it need not be between zero and one.

In [11]:
def log_likelihood(data, mean, variance):
    """Joint log-density for independent 1D Gaussian observations."""
    data = np.asarray(data, dtype=float)
    mean = np.asarray(mean, dtype=float)
    if data.ndim != 1 or len(data) < 1 or not np.isfinite(data).all():
        raise ValueError("data must be a finite, nonempty one-dimensional array.")
    if mean.ndim != 0 or not np.isfinite(mean).all():
        raise ValueError("mean must be a finite scalar.")
    if not np.isfinite(variance) or variance <= 0:
        raise ValueError("variance must be positive and finite.")
    squared_distances = np.sum((data - mean) ** 2)
    return float(-len(data) / 2 * np.log(2 * np.pi * variance)
                 - squared_distances / (2 * variance))


def scientific_likelihood(log_value):
    """Format exp(log_value) without underflowing the likelihood to zero."""
    log10_value = log_value / np.log(10)
    exponent = int(np.floor(log10_value))
    mantissa = 10 ** (log10_value - exponent)
    return f"{mantissa:.4f} × 10^({exponent})"


mle_mean = float(points.mean())
mle_variance = float(np.sum((points - mle_mean) ** 2) / len(points))
if mle_variance <= 0:
    raise ValueError("Use a sample with nonzero spread for a finite positive-variance MLE.")
mle_log_likelihood = log_likelihood(points, mle_mean, mle_variance)


def gaussian_density(x, mean, variance):
    return np.exp(-0.5 * (np.asarray(x) - mean) ** 2 / variance) / np.sqrt(2 * np.pi * variance)


In [12]:
# Keep both axes and histogram bins fixed while exploring.
plot_low = float(min(points.min(), 0) - 2)
plot_high = float(max(points.max(), 0) + 2)
histogram_edges = np.histogram_bin_edges(points, bins="auto")
histogram_density, _ = np.histogram(points, bins=histogram_edges, density=True)
# Leave headroom above the histogram and MLE peak; hold this limit fixed.
y_max = 1.25 * max(histogram_density.max(), gaussian_density(mle_mean, mle_mean, mle_variance))
x_grid = np.linspace(plot_low, plot_high, 1200)
slider_style = {"description_width": "100px"}
slider_layout = widgets.Layout(width="550px")
mean_control = widgets.FloatSlider(
    value=0, min=plot_low, max=plot_high, step=0.05, description="Mean μ:",
    continuous_update=False, readout_format=".3f", style=slider_style, layout=slider_layout,
)
variance_control = widgets.FloatLogSlider(
    value=2 * mle_variance, base=10, min=np.log10(mle_variance / 50),
    max=np.log10(mle_variance * 20), step=0.02, description="Variance σ²:",
    continuous_update=False, readout_format=".4f", style=slider_style, layout=slider_layout,
)
show_mle = widgets.Checkbox(value=False, description="Show MLE")
fit_button = widgets.Button(description="Set sliders to MLE", button_style="info", disabled=True)
output = widgets.Output()
updating = False


def render_fit(change=None):
    if updating:
        return
    mean = mean_control.value
    variance = variance_control.value
    sigma = np.sqrt(variance)
    ll = log_likelihood(points, mean, variance)
    with output:
        output.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.hist(points, bins=histogram_edges, density=True, color="#A8BDCC",
                edgecolor="white", alpha=0.8, label=f"Observed density (n={len(points)})")
        # Add a fine grid around the chosen mean so even narrow curves are smooth.
        x = np.unique(np.concatenate([x_grid, np.linspace(mean - 5 * sigma, mean + 5 * sigma, 600)]))
        ax.plot(x, gaussian_density(x, mean, variance), color="#C65319", linewidth=2.5,
                label="Chosen Gaussian")
        ax.plot(points, np.full(len(points), 0.025), "|", color="#304B63", markersize=12,
                transform=ax.get_xaxis_transform(), label="Individual observations")
        ax.axvline(mean, color="#C65319", linestyle=":", alpha=0.7, label="Chosen mean")
        if show_mle.value:
            ax.plot(x_grid, gaussian_density(x_grid, mle_mean, mle_variance),
                    color="#287D63", linestyle="--", linewidth=2, label="MLE Gaussian")
        ax.set(xlim=(plot_low, plot_high), ylim=(0, y_max),
               xlabel="Observed value x", ylabel="Probability density",
               title="Fit a Gaussian to the fixed sample")
        ax.grid(alpha=0.15)
        ax.legend(loc="upper right", fontsize=9)
        fig.tight_layout()
        plt.show()
        plt.close(fig)
        display(Markdown(
            f"**Log-likelihood: {ll:.4f}** &nbsp; | &nbsp; "
            f"Likelihood: `{scientific_likelihood(ll)}`\n\n"
            f"Mean μ = {mean:.4f}; "
            f"variance σ² = {variance:.4f}; standard deviation σ = {sigma:.4f}."
        ))
        if show_mle.value:
            gap = max(0.0, mle_log_likelihood - ll)
            display(Markdown(
                f"**MLE:** μ = {mle_mean:.4f}, "
                f"σ² = {mle_variance:.4f}. Maximum log-likelihood = {mle_log_likelihood:.4f}.\n\n"
                f"**Log-likelihood gap to the maximum: {gap:.4f}** (zero is best)."
            ))


def toggle_mle(change):
    fit_button.disabled = not change["new"]
    render_fit()


def set_to_mle(_):
    global updating
    updating = True
    try:
        mean_control.value = mle_mean
        variance_control.value = mle_variance
    finally:
        updating = False
    render_fit()


for control in [mean_control, variance_control]:
    control.observe(render_fit, names="value")
show_mle.observe(toggle_mle, names="value")
fit_button.on_click(set_to_mle)
display(widgets.VBox([
    mean_control, variance_control,
    widgets.HBox([show_mle, fit_button]),
    widgets.HTML("Both axes and histogram bins stay fixed; very narrow Gaussian peaks may extend above the plot."),
    output,
]))
render_fit()
